In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()


df = spark.read.table("corporate_data_lakehouse.bronze.comp_house_people")

df.display()

***Most Important functions
- from_json-->string -> to convert structured column
- Explode --> raw  --> multiple rows
- Explode_outer --> same as expolede but handle if data is null.

In [0]:
from pyspark.sql.functions import explode_outer, col, split, lpad, get, trim, when, size

#explode_outer used to explode the officer details into multiple rows

df_exploded_of = (
    df.withColumn("items", explode_outer("items"))
      .withColumn(
            "company_number", 
            split(col("links.self"), "/")[2])
)

print("exploded_items completed ")

#split_name used to seprate the officer name into last name, given names and title
split_name = split(col("items.name"), ",")    

#selecting the required columns

df_officer_details = df_exploded_of.select(
        "company_number",
        trim(get(split_name, 2)).alias("title"),
        trim(get(split_name, 1)).alias("first_names"),
        trim(get(split_name, 0)).alias("last_name"), 
        col("items.person_number").alias("officer_id"),
        "items.officer_role", 
        lpad(col("items.date_of_birth.month").cast("string"), 2, "0").alias("birth_month"), #lpad & cast used for to get '0' in month
        col("items.date_of_birth.year").alias("birth_year"),  
        "items.nationality",
        "items.address.address_line_1", 
        "items.address.premises", 
        col("items.address.locality").alias("city"), 
        "items.address.region", 
        "items.address.country",
        "items.address.postal_code", 
        "items.appointed_on", 
        "items.resigned_on"
)
print("officer_colums generated")

#df_officer_details.display()
print("officer_details saving")
df_officer_details.write.mode("overwrite").saveAsTable("corporate_data_lakehouse.silver.officers")

print("officer table saved")

In [0]:
'''process:
import sparksession
created spark session
read table

avoided from_json bcoz data already structed in json
used Explode_outer on item column 
split company number from links column

there was cvs data in officers_name split into title, name, last name here with help of get()handled null values
in d.o.b lpad n cast used to to get month as 00,01,02,03 from 1,2,3'''
